# 03 · Feature signals

Candidate features for the ranker (spec 9) and the towers, each checked on its own:

- **A. Raw review fields** the marts drop (playtime, ownership, language, purchase, …) vs the
  label. Check *when* each was measured: fields as of the scrape leak the future.
- **B. User history** (strictly before the review) vs the label.
- **C. Next-game discrimination**: for positive reviews with history, the reviewed game vs random
  catalog games: affinity and fit features, scored by single-feature AUC.

Single-feature AUC is a screening tool, not the answer: correlated features, interactions and the
retriever's own score decide in the ranker.

In [ ]:
import os

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns

from steam_eda import (
    Sample,
    binned_rate,
    cached,
    categorical_rate,
    decode,
    feature_report,
    game_catalog,
    gini,
    iter_mart,
    load_mart,
    load_raw,
    lookup,
    lorenz,
    mart_schema,
    mart_table,
    raw_files,
    review_counts,
    tag_profile_similarity,
    with_user_history,
)
from steam_eda.plots import loglog_hist, rate_plot, setup

setup()

SAMPLE_USERS = float(os.environ.get("EDA_SAMPLE_USERS", 0.01))
RAW_LAST_FILES = (
    int(os.environ["EDA_RAW_LAST_FILES"]) if os.environ.get("EDA_RAW_LAST_FILES") else None
)
NEGATIVES_PER_POSITIVE = 4
# "popularity": negatives drawn ∝ positive reviews (like in-batch negatives, closer to what the
# retriever returns); "uniform": any reviewed game (popularity alone then separates almost all)
NEGATIVE_SAMPLING = os.environ.get("EDA_NEGATIVE_SAMPLING", "popularity")
catalog = cached("catalog", game_catalog)

## A. Raw review fields vs sentiment

`playtime_at_review` and the review flags are measured at review time. `playtime_forever`,
`num_games_owned`, `num_reviews` and `votes_*` are measured **at scrape time**, after the review,
so they're only safe as rough user traits (and leak in a time split).

In [ ]:
raw = load_raw(
    "reviews",
    columns=[
        "rec_id",
        "author_id",
        "voted_up",
        "playtime_at_review",
        "playtime_forever",
        "num_games_owned",
        "num_reviews",
        "language",
        "steam_purchase",
        "received_for_free",
        "written_during_early_access",
        "primarily_steam_deck",
        "votes_up",
        "weighted_vote_score",
    ],
    sample=Sample.by_key("author_id", SAMPLE_USERS),
    last_files=RAW_LAST_FILES,
).unique("rec_id")
raw = raw.with_columns(
    log_playtime_at_review=(pl.col("playtime_at_review") + 1).log10(),
    log_num_games_owned=(pl.col("num_games_owned") + 1).log10(),
)
print(f"{raw.height:,d} raw reviews of {raw['author_id'].n_unique():,d} authors")
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
rate_plot(
    binned_rate(raw, "log_playtime_at_review", "voted_up", bins=20),
    "voted_up vs log10 playtime at review (min)",
    ax=axes[0],
)
rate_plot(
    binned_rate(raw, "log_num_games_owned", "voted_up", bins=20),
    "voted_up vs log10 games owned (scrape time)",
    ax=axes[1],
)
rate_plot(
    binned_rate(raw, "num_reviews", "voted_up", bins=20),
    "voted_up vs author's reviews (scrape time)",
    ax=axes[2],
)
axes[2].set_xscale("symlog")

In [ ]:
flags = [
    "steam_purchase",
    "received_for_free",
    "written_during_early_access",
    "primarily_steam_deck",
]
pl.concat(
    [
        categorical_rate(raw, f, "voted_up", min_rows=50)
        .rename({f: "value"})
        .with_columns(pl.col("value").cast(pl.String), feature=pl.lit(f))
        for f in flags
    ]
).select("feature", "value", "rows", "rate", "lift")

In [ ]:
categorical_rate(raw, "language", "voted_up", min_rows=500).head(20)

## B. User history vs sentiment (marts, strictly before each review)

In [ ]:
inter = with_user_history(
    load_mart(
        "interactions",
        columns=[
            "user_id",
            "game_idx",
            "timestamp",
            "is_positive",
            "game_reviews_ratio",
            "games_reviewed_positive",
        ],
        sample=Sample.by_key("user_id", SAMPLE_USERS),
    )
).join(
    catalog.select(
        "game_idx",
        "game_price",
        "release_year",
        "game_is_free",
        pl.col("positive_reviews").alias("game_positive_reviews"),
    ),
    on="game_idx",
    how="left",
)
inter = inter.with_columns(log_game_popularity=(pl.col("game_positive_reviews") + 1).log10())
feature_report(
    inter,
    [
        "prior_positive_rate",
        "prior_reviews",
        "days_since_prior_review",
        "game_reviews_ratio",
        "log_game_popularity",
        "game_price",
        "release_year",
    ],
    label="is_positive",
)

`game_reviews_ratio` is the game's ratio *as of before the review* (time-versioned in
`game_features`), so it is leakage-safe; `log_game_popularity` and `game_price` here use the
**current** catalog (lifetime counts, today's price), so they are optimistic. Build them as of t
for training.

## C. Next-game discrimination: reviewed game vs random games

For each positive review with history (`games_reviewed_positive`, the last ≤5 positives before
it), label 1 = the reviewed game, label 0 = `NEGATIVES_PER_POSITIVE` reviewed, released catalog games
drawn by `NEGATIVE_SAMPLING`. Features compare the candidate with the user's history. Sampled
negatives are easier than the retriever's top 100 (what spec 9 ranks), so expect AUCs above what
the ranker will see; the **order** of features is what to read. With popularity sampling, the
popularity features measure the bias *beyond* popularity.

In [ ]:
import numpy as np

pool = catalog.filter(
    (pl.col("positive_reviews") > 0) & ~pl.col("game_coming_soon").fill_null(False)
)
weights = (
    pool["positive_reviews"].to_numpy().astype(float) if NEGATIVE_SAMPLING == "popularity" else None
)
queries = (
    inter.filter(pl.col("is_positive"))
    # drop the padding (0) and OOV (1) ids of the last-5 history
    .with_columns(pl.col("games_reviewed_positive").list.filter(pl.element() > 1))
    .filter(pl.col("games_reviewed_positive").list.len() > 0)
    .with_row_index("query")
    .select(
        "query",
        "user_id",
        "timestamp",
        "games_reviewed_positive",
        pl.col("game_idx").alias("target"),
    )
)
rng = np.random.default_rng(0)
drawn = rng.choice(
    pool["game_idx"].to_numpy(),
    size=queries.height * NEGATIVES_PER_POSITIVE,
    p=None if weights is None else weights / weights.sum(),
)
negatives = (
    queries.select(
        pl.col("query")
        .repeat_by(NEGATIVES_PER_POSITIVE)
        .list.explode(keep_nulls=False, empty_as_null=False)
    )
    .with_columns(candidate=pl.Series(drawn), label=pl.lit(False))
    .join(queries.drop("target"), on="query")
)
pairs = pl.concat(
    [
        queries.rename({"target": "candidate"}).with_columns(label=pl.lit(True)),
        negatives.select(
            "query", "user_id", "timestamp", "games_reviewed_positive", "candidate", "label"
        ),
    ]
).filter(~pl.col("games_reviewed_positive").list.contains(pl.col("candidate")))
print(f"{queries.height:,d} queries, {pairs.height:,d} pairs")

In [ ]:
game = catalog.select(
    "game_idx",
    "game_tags",
    "game_genres",
    "game_developers",
    "game_publishers",
    "game_price",
    "release_year",
    "game_reviews_ratio",
    "game_is_free",
    log_popularity=(pl.col("positive_reviews") + 1).log10(),
)
history = (
    pairs.select("query", "games_reviewed_positive")
    .unique("query")
    .explode("games_reviewed_positive", empty_as_null=False)
    .rename({"games_reviewed_positive": "game_idx"})
    .filter(pl.col("game_idx") > 1)
    .join(game, on="game_idx", how="left")
    .group_by("query")
    .agg(
        h_tags=pl.col("game_tags")
        .list.head(5)
        .list.explode(keep_nulls=False, empty_as_null=False)
        .unique(),
        h_genres=pl.col("game_genres").list.explode(keep_nulls=False, empty_as_null=False).unique(),
        h_developers=pl.col("game_developers").list.explode(keep_nulls=False, empty_as_null=False),
        h_publishers=pl.col("game_publishers").list.explode(keep_nulls=False, empty_as_null=False),
        h_popularity=pl.col("log_popularity").mean(),
        h_year=pl.col("release_year").median(),
        h_price=pl.col("game_price").median(),
        h_len=pl.len(),
    )
)
scored = pairs.join(history, on="query", how="left").join(
    game.rename(lambda c: c if c == "game_idx" else f"c_{c.removeprefix('game_')}"),
    left_on="candidate",
    right_on="game_idx",
    how="left",
)
scored = scored.with_columns(
    tag_jaccard=tag_profile_similarity(scored["h_tags"], scored["c_tags"].list.head(5)),
    genre_jaccard=tag_profile_similarity(scored["h_genres"], scored["c_genres"]),
    developer_share=pl.col("h_developers").list.set_intersection("c_developers").list.len()
    / pl.col("h_len"),
    publisher_share=pl.col("h_publishers").list.set_intersection("c_publishers").list.len()
    / pl.col("h_len"),
    popularity=pl.col("c_log_popularity"),
    popularity_gap=(pl.col("c_log_popularity") - pl.col("h_popularity")).abs(),
    year_gap=(pl.col("c_release_year") - pl.col("h_year")).abs(),
    price_gap=(pl.col("c_price") - pl.col("h_price")).abs(),
    reviews_ratio=pl.col("c_reviews_ratio"),
)
CANDIDATES = [
    "tag_jaccard",
    "genre_jaccard",
    "developer_share",
    "publisher_share",
    "popularity",
    "popularity_gap",
    "year_gap",
    "price_gap",
    "reviews_ratio",
]
report = feature_report(scored, CANDIDATES, label="label")
report

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, feature in zip(axes, report["feature"].head(3), strict=True):
    sns.kdeplot(
        data=scored.select(feature, "label").drop_nulls().to_pandas(),
        x=feature,
        hue="label",
        common_norm=False,
        ax=ax,
    )
    ax.set_title(feature)

## Shortlist

| Feature | Signal (AUC / lift) | Leakage-safe as of t? | Available at inference? | Verdict |
|---|---|---|---|---|
|  |  |  |  |  |

Leakage-safe: computed only from data before the review. Available at inference: the batch path
has full history; the online path (`POST /recommendations`) only has the liked games.